Extract and store generalized facts from conversations, building a persistent knowledge base that transcends any single interaction.

You know that Paris is the capital of France. But you probably can't remember the exact moment you learned it. That's semantic memory at work: knowledge distilled from experience into stable, context-free facts. The "when" and "how" fade away. The fact itself remains.

AI agents face the same challenge. A user mentions their favorite language is Python in session one, their team size in session two, and their deployment target in session three. Without semantic memory, each session starts from zero. With it, the agent builds a growing profile of distilled facts. Over time it stops asking the same questions. It anticipates needs based on what it already knows.

This notebook shows you how to build a semantic memory system from scratch using the OpenAI SDK. You'll extract facts from conversations with an LLM call, store them with vector embeddings (numerical representations of meaning), detect duplicates and contradictions, and retrieve relevant facts for future prompts.

By the end you'll understand:

How to extract clean, declarative facts from messy conversation text.

How embedding similarity catches duplicates that string matching misses.

How to detect and resolve contradictions when users change their minds.

When semantic memory helps and when it quietly fails.

### Key Concepts

Semantic memory: A store of general facts and knowledge, separate from specific episodes. In cognitive science, Endel Tulving coined this term in 1972 to distinguish "knowing that" from "remembering when."

Fact extraction: Pulling declarative statements from conversation text using an LLM prompt. "Oh yeah, I switched to a Mac last month" becomes the fact: "User uses macOS."

Embedding: A list of numbers (a vector) that captures the meaning of a piece of text. Texts with similar meaning have embeddings that point in similar directions. We use embeddings to compare facts by meaning, not by exact wording.

Cosine similarity: A measure of how similar two vectors are. It ranges from -1 (opposite) to 1 (identical direction). We use it to compare embeddings.

Deduplication: Detecting when a new fact means the same thing as one already stored. "I use a Mac" and "My laptop runs macOS" are the same fact. Cosine similarity catches this even though the words differ.

Contradiction detection: Identifying when a new fact conflicts with an existing one. "I use Windows" contradicts a stored "User uses macOS." The system must decide which to keep (usually the newer one).

Confidence score: A number from 0.0 to 1.0 that represents how certain we are about a fact. Explicit statements get high confidence. Inferred facts get lower confidence. Repeated mentions boost confidence.

Context injection: Inserting retrieved facts into the system prompt so the LLM can use them when generating a response.


> **The memory type progression:**
>
> | Type | What it stores | When was it true? | Example |
> |---|---|---|---|
> | Episodic (09) | "On June 12, Chiru was anxious about markets and chose debt funds" | Specific date | Diary entry |
> | **Semantic (10)** | **"Chiru consistently panics during market volatility and needs reassurance before deciding"** | **Always — independent of when learned** | **Encyclopedia entry** |
>
> Episodic memory records what happened. Semantic memory extracts what it means — general truths that hold across all episodes.

---

## What Is Semantic Memory?

### The core idea in one sentence
Distil general, reusable facts and behavioural patterns from raw episodic records — strip away the "when" and "where" and keep only the durable truth.

---

### The mental model — a distillery

Episodic memory is the raw ingredients: every session, every message, every event. Semantic memory is the distillate — the concentrated essence that survives across all sessions and remains useful forever.

```
RAW EPISODES (episodic memory):
  Session 1: "Chiru asked about FD options, seemed anxious, chose lowest-risk option"
  Session 3: "Market dip mentioned — Chiru immediately asked to move to FD"
  Session 5: "Chiru hesitated on equity SIP — cited fear of loss — ultimately declined"
  Session 7: "Chiru asked about debt fund safety during rising rate environment"
                              ↓
              DISTILLATION PROCESS
                              ↓
SEMANTIC FACTS:
  "Chiru has strong loss aversion — consistently prioritises capital preservation over returns"
  "Market volatility triggers anxiety in Chiru — requires reassurance before any decision"
  "Chiru's de facto risk profile is conservative regardless of stated financial goals"
  "Chiru prefers to discuss debt instruments — consistently deflects equity discussions"
```

These facts are not tied to any specific session. They are general truths derived from the pattern across many sessions. They are what makes the agent feel experienced — not just informed.

---

### How it differs from episodic memory

| | Episodic (09) | Semantic (10) |
|---|---|---|
| Unit stored | Complete session record | Individual distilled fact |
| Time-bound | Yes — "on June 12th" | No — "always true" |
| Source | One session | Pattern across many sessions |
| Immutable? | Yes — never rewritten | No — facts evolve as patterns change |
| Answers | "What happened in April?" | "What is generally true about this user?" |
| Token cost | Medium (episode summaries) | Low (compact fact statements) |

---

### How it differs from entity memory

Entity Memory (07) stores structured **current-state facts**: `salary = ₹1,20,000`, `risk_profile = conservative`. These are precise, schema-bound, updated in-place.

Semantic Memory stores **behavioural and contextual generalisations**: "Chiru consistently panics during market volatility." These are fuzzy, free-form, derived from observed patterns — not from a single stated fact.

| | Entity Memory (07) | Semantic Memory (10) |
|---|---|---|
| Format | Structured key-value | Free-form natural language |
| Source | What user explicitly stated | Pattern inferred from behaviour |
| Update | In-place replacement | Accumulates and evolves |
| Example | `salary: ₹1,20,000` | "user avoids equity discussions" |

---

### Point 1 — Semantic facts are distilled from episodes, not extracted from messages

The distillation process is the heart of this technique. After accumulating multiple episodes, we run a distillation call that reads all episodes and asks: "What general patterns are true about this user across all these sessions?" The output is a set of compact, reusable semantic facts.

This is why episodic memory (Technique 09) is the natural prerequisite — semantic memory is the next layer up in the abstraction hierarchy.

---

### Point 2 — Semantic facts are categorised for targeted injection

Not all semantic facts are relevant to every query. We categorise them:

- **behavioural**: how the user typically acts (anxiety patterns, decision styles)
- **financial**: generalised financial patterns (saving habits, spending tendencies)
- **preference**: what the user consistently likes or dislikes
- **risk**: risk-related patterns observed across sessions
- **communication**: how the user likes to receive advice

At injection time, we can filter by category — injecting only the relevant categories for the current query type.

---

### Point 3 — Facts have a confidence score and observation count

A fact observed once is a hypothesis. A fact confirmed across five sessions is a reliable generalisation. We track:
- `observation_count`: how many sessions contributed to this fact
- `confidence`: a score from 0 to 1 reflecting reliability
- `last_confirmed`: when this pattern was last observed

In production, only facts above a confidence threshold are injected. Low-confidence facts are stored but not surfaced until confirmed.

---

### Point 4 — Semantic facts evolve — they can strengthen, weaken, or be contradicted

Unlike episodic records (immutable) or entity values (in-place replacement), semantic facts evolve gradually. A new episode that contradicts a fact reduces its confidence. A new episode that confirms it increases confidence and updates `last_confirmed`.

Example: if Chiru suddenly starts enthusiastically asking about equity SIPs, the fact "Chiru consistently deflects equity discussions" should weaken — not be immediately deleted, but confidence should drop and a new competing fact emerges.

---

### Point 5 — Semantic memory is the cheapest long-term memory layer per token

A well-maintained semantic memory store for a user might be 300-500 tokens — a compact set of general truths. Compare this to injecting 3 full episode summaries (~750 tokens) or a vector store retrieval set (~600 tokens). Semantic facts give the highest signal-to-token ratio of any long-term memory type.

---

## The Distillation Pipeline

```
Episodes accumulate
       ↓
Distillation trigger fires (e.g. every 3 new episodes)
       ↓
All episodes passed to distillation LLM call
       ↓
Structured semantic facts returned as JSON
       ↓
Facts merged into semantic store
  - New fact: add with confidence = 0.6 (single observation)
  - Existing fact confirmed: confidence += 0.1 (up to 1.0)
  - Existing fact contradicted: confidence -= 0.2
  - Confirmed fact: update last_confirmed timestamp
       ↓
At inference time: inject facts above confidence threshold
```

---

## Trade-offs

| | |
|---|---|
| ✅ Highest signal-to-token ratio of any memory type | ❌ Requires multiple episodes to produce reliable facts |
| ✅ General patterns — useful across all query types | ❌ Distillation adds cost (extra LLM call per batch) |
| ✅ Compact — 300-500 tokens for a full user profile | ❌ Facts can be wrong if early episodes are misleading |
| ✅ Enables behavioural personalisation at scale | ❌ No temporal tracking — when did this pattern start? |
| ✅ Evolves with the user over time | ❌ Hard to audit — where did this fact come from? |

## Production Verdict

> **The most token-efficient long-term memory. Build it on top of episodic memory. Inject it always.**
>
> Semantic memory is what separates a smart agent from a wise one. An agent with episodic memory remembers what happened. An agent with semantic memory understands the user. For FinCoach, the difference is between recalling "Chiru chose debt funds in April" and knowing "Chiru will always choose capital preservation over returns when under pressure." The second produces fundamentally better advice.

---

In [1]:
# Install required packages.
# 'openai'   — GPT-4o for conversation + semantic distillation.
# 'tiktoken' — exact token counting.
!pip install openai tiktoken --quiet

In [2]:
# --- Standard library ---
import json                              # Serialisation and JSON response parsing.
import os                                # Environment variables.
import time                              # Rate-limit delays.
from datetime import datetime, timezone  # UTC timestamps.
from typing import List, Dict, Optional, Any
from dataclasses import dataclass, field, asdict
from enum import Enum

# --- Third-party ---
import openai    # OpenAI SDK.
import tiktoken  # Exact tokeniser for GPT-4o.

In [3]:
# --- API Key Setup ---
try:
    from google.colab import userdata
    OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
    print("Key loaded from Colab Secrets.")
except Exception:
    OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "")
    print("Key loaded from environment variable.")

assert OPENAI_API_KEY and OPENAI_API_KEY.startswith("sk-"), \
    "API key missing or invalid."

client = openai.OpenAI(api_key=OPENAI_API_KEY)

CHAT_MODEL        = "gpt-4o"
# Main conversation model.

DISTILLATION_MODEL = "gpt-4o-mini"
# Used for semantic distillation — pattern extraction from episodes.
# Cheaper model is sufficient; this is a summarisation/extraction task.

TOKENISER = tiktoken.get_encoding("o200k_base")

print(f"Chat model        : {CHAT_MODEL}")
print(f"Distillation model: {DISTILLATION_MODEL}")

Key loaded from environment variable.
Chat model        : gpt-4o
Distillation model: gpt-4o-mini


---
## Part 1 — The Semantic Fact Data Model

A semantic fact is a general truth about the user — with confidence tracking, category, source evidence, and evolution history.

In [4]:
class FactCategory(str, Enum):
    """
    Categories for semantic facts.
    Enables targeted injection — only inject categories relevant to the current query.
    """
    BEHAVIOURAL   = "behavioural"   # How the user typically acts and decides.
    FINANCIAL     = "financial"     # General financial patterns and tendencies.
    RISK          = "risk"          # Risk-related patterns observed across sessions.
    PREFERENCE    = "preference"    # What the user consistently likes or dislikes.
    COMMUNICATION = "communication" # How the user prefers to receive advice.
    GENERAL       = "general"       # Facts that do not fit other categories.


@dataclass
class SemanticFact:
    """
    A single distilled general truth about a user.
    Derived from patterns observed across multiple episodes.
    """

    fact_id: str
    # Unique identifier — used for deduplication and updating.
    # Generated as a short hash of the fact statement for natural deduplication.

    user_id: str
    # The user this fact belongs to.

    statement: str
    # The fact itself — a single sentence in natural language.
    # Example: "Chiru consistently prioritises capital preservation over returns."
    # Must be general (no specific dates or session references).

    category: str
    # One of the FactCategory values.
    # Used for filtering at injection time.

    confidence: float
    # How reliable this fact is, from 0.0 to 1.0.
    # 0.6: observed once — hypothesis
    # 0.8: confirmed in 2-3 sessions — likely true
    # 0.95: confirmed in 5+ sessions — reliable generalisation
    # Below 0.4: weakening — may be contradicted

    observation_count: int
    # How many episodes contributed to this fact.
    # Low count (1-2): treat with caution.
    # High count (5+): high reliability.

    source_episode_ids: List[str]
    # Which episode IDs contributed to this fact.
    # Audit trail: "why does the agent believe this?"

    first_observed: str
    # When this fact was first extracted.

    last_confirmed: str
    # When this fact was most recently confirmed or updated.
    # Old last_confirmed = potentially stale — may need re-evaluation.

    evolution_log: List[Dict] = field(default_factory=list)
    # Log of every change to this fact:
    # [{"action": "created"|"confirmed"|"weakened"|"contradicted",
    #   "old_confidence": 0.7, "new_confidence": 0.8,
    #   "timestamp": "..." }]
    # Audit trail for fact evolution.

    def is_reliable(self, threshold: float = 0.7) -> bool:
        """Return True if this fact meets the confidence threshold for injection."""
        return self.confidence >= threshold

    def strengthen(self, episode_id: str) -> None:
        """
        Increase confidence when a new episode confirms this fact.
        Confidence grows by 0.1 per confirmation, capped at 1.0.
        """
        old_confidence = self.confidence
        self.confidence = min(1.0, self.confidence + 0.1)
        # Cap at 1.0 — maximum certainty.
        self.observation_count += 1
        if episode_id not in self.source_episode_ids:
            self.source_episode_ids.append(episode_id)
        self.last_confirmed = datetime.now(timezone.utc).isoformat()
        self.evolution_log.append({
            "action": "confirmed",
            "old_confidence": round(old_confidence, 3),
            "new_confidence": round(self.confidence, 3),
            "episode_id": episode_id,
            "timestamp": self.last_confirmed,
        })

    def weaken(self, reason: str = "") -> None:
        """
        Decrease confidence when a new episode contradicts this fact.
        Confidence drops by 0.2 per contradiction.
        """
        old_confidence = self.confidence
        self.confidence = max(0.0, self.confidence - 0.2)
        # Floor at 0.0 — cannot have negative confidence.
        now = datetime.now(timezone.utc).isoformat()
        self.evolution_log.append({
            "action": "weakened",
            "old_confidence": round(old_confidence, 3),
            "new_confidence": round(self.confidence, 3),
            "reason": reason,
            "timestamp": now,
        })

    def format_for_injection(self) -> str:
        """Format as a single line for injection into the API context."""
        conf_label = (
            "[high confidence]" if self.confidence >= 0.85 else
            "[medium confidence]" if self.confidence >= 0.65 else
            "[low confidence]"
        )
        return f"- {self.statement} {conf_label}"


print("SemanticFact dataclass defined.")

SemanticFact dataclass defined.


---
## Part 2 — The Distillation Engine

The distillation engine reads a set of episode summaries and extracts general semantic facts. This is where episodic memory becomes semantic memory.

In [5]:
DISTILLATION_PROMPT = """You are a semantic memory distiller for FinCoach.
Your job is to read a set of past session summaries and extract GENERAL TRUTHS about the user.

These are not facts about specific sessions (that is episodic memory).
These are patterns that hold across multiple sessions — reusable generalisations.

Return a JSON object with this structure:
{
  "facts": [
    {
      "statement": "A single general truth about the user — no specific dates or sessions",
      "category": "behavioural | financial | risk | preference | communication | general",
      "confidence": 0.6 to 0.9 (based on how consistently this pattern appears)
    }
  ]
}

CATEGORIES:
- behavioural : how the user typically acts and makes decisions
- financial   : general financial patterns and tendencies
- risk        : risk tolerance and risk-related patterns
- preference  : what the user consistently prefers or avoids
- communication: how the user likes to receive and process advice
- general     : anything that does not fit the above

CONFIDENCE GUIDANCE:
- 0.6 : pattern observed in 1 session — possible but uncertain
- 0.7 : pattern in 2 sessions — likely
- 0.8 : pattern in 3+ sessions — reliable
- 0.9 : consistent across all sessions — strong generalisation

STRICT RULES:
1. Return ONLY valid JSON — no markdown, no explanation.
2. Each statement must be general — no specific dates, session numbers, or amounts.
3. Extract only patterns that genuinely appear across episodes — do not invent.
4. Maximum 8 facts — choose the most useful and reliable ones.
5. Each fact must be actionable — it should change how FinCoach advises this user.
6. Write in third person: 'The user...' or 'Chiru...'
"""


def distil_episodes_to_facts(
    episode_summaries: List[Dict],
    user_id: str,
) -> List[SemanticFact]:
    """
    Extract semantic facts from a list of episode summaries.
    This is the distillation step: many episodes → compact general truths.

    Args:
        episode_summaries: List of dicts with 'summary', 'date', 'episode_id'.
        user_id:           The user these episodes belong to.

    Returns:
        List of SemanticFact objects ready for merging into the semantic store.
    """

    # Format episode summaries for the distillation prompt.
    episodes_text = "\n\n".join(
        f"[Session {i+1} — {ep.get('date', 'unknown')}]:\n{ep['summary']}"
        for i, ep in enumerate(episode_summaries)
    )
    # Number the sessions to help the model assess pattern frequency.

    response = client.chat.completions.create(
        model=DISTILLATION_MODEL,
        # gpt-4o-mini: pattern extraction task — cheaper model is sufficient.
        max_tokens=800,
        # 8 facts at ~50 tokens each = 400 tokens max — 800 gives headroom.
        temperature=0.0,
        # Deterministic — fact extraction should not vary.
        response_format={"type": "json_object"},
        # Guaranteed valid JSON — no parsing issues.
        messages=[
            {"role": "system", "content": DISTILLATION_PROMPT},
            {"role": "user",   "content":
             f"Extract general semantic facts from these {len(episode_summaries)} sessions:\n\n"
             f"{episodes_text}"
            },
        ]
    )

    raw = json.loads(response.choices[0].message.content)
    raw_facts = raw.get("facts", [])

    # Convert raw dicts to SemanticFact objects.
    facts = []
    episode_ids = [ep.get("episode_id", "") for ep in episode_summaries]
    now = datetime.now(timezone.utc).isoformat()

    for raw_fact in raw_facts:
        statement  = raw_fact.get("statement", "").strip()
        if not statement:
            continue
            # Skip empty statements — safety guard.

        # Generate a short hash as the fact_id for deduplication.
        import hashlib
        fact_id = hashlib.md5(
            f"{user_id}:{statement[:80]}".encode()
        ).hexdigest()[:12]
        # MD5 of (user_id + first 80 chars of statement).
        # Same statement for same user → same ID → natural deduplication.

        fact = SemanticFact(
            fact_id=fact_id,
            user_id=user_id,
            statement=statement,
            category=raw_fact.get("category", FactCategory.GENERAL),
            confidence=float(raw_fact.get("confidence", 0.6)),
            observation_count=len(episode_summaries),
            # All provided episodes contributed to each extracted fact.
            source_episode_ids=episode_ids,
            first_observed=now,
            last_confirmed=now,
        )
        facts.append(fact)

    tokens_used = response.usage.total_tokens
    print(f"  [DISTIL] {len(episode_summaries)} episodes → "
          f"{len(facts)} semantic facts | tokens: {tokens_used}")

    return facts


print("Distillation engine defined.")

Distillation engine defined.


---
## Part 3 — The SemanticMemoryStore

Manages the full lifecycle of semantic facts: storing, updating, merging, and querying.

In [6]:
class SemanticMemoryStore:
    """
    Persistent store for a user's semantic memory facts.

    Facts are stored in-memory (dict keyed by fact_id) and persisted to JSON.
    In production: store in a relational DB (facts table) with vector embeddings
    for semantic similarity deduplication.

    The store handles:
    - Inserting new facts (from distillation)
    - Merging similar facts (avoid duplicates)
    - Strengthening facts when re-confirmed
    - Weakening facts when contradicted
    - Querying by category, confidence, and relevance
    """

    def __init__(self, user_id: str, confidence_threshold: float = 0.65):
        self.user_id              = user_id
        self.confidence_threshold = confidence_threshold
        # Facts below this threshold are stored but not injected.
        # 0.65: inject facts confirmed in at least 2 sessions.

        self._facts: Dict[str, SemanticFact] = {}
        # Key: fact_id, Value: SemanticFact object.
        # Hash-based deduplication: same fact_id = same fact.

        self._distillation_count = 0
        # How many distillation runs have been done.

        self.created_at = datetime.now(timezone.utc).isoformat()

        print(f"[SemanticStore] Initialised for user: {self.user_id}")

    def merge_facts(self, new_facts: List[SemanticFact]) -> Dict[str, int]:
        """
        Merge a list of newly distilled facts into the store.

        For each new fact:
        - If fact_id exists: strengthen (same fact confirmed again)
        - If fact_id is new: add as a new fact

        Returns a summary dict: {added: N, strengthened: N}
        """
        added       = 0
        strengthened = 0

        for fact in new_facts:
            if fact.fact_id in self._facts:
                # Fact already exists — same pattern observed again.
                # Strengthen it: confidence += 0.1, update last_confirmed.
                episode_id = fact.source_episode_ids[-1] if fact.source_episode_ids else ""
                self._facts[fact.fact_id].strengthen(episode_id)
                strengthened += 1
            else:
                # New fact — add to store.
                self._facts[fact.fact_id] = fact
                added += 1

        self._distillation_count += 1

        print(f"  [MERGE] Distillation #{self._distillation_count}: "
              f"+{added} new facts, {strengthened} strengthened")
        return {"added": added, "strengthened": strengthened}

    def get_reliable_facts(
        self,
        categories: Optional[List[str]] = None,
        min_confidence: Optional[float] = None,
    ) -> List[SemanticFact]:
        """
        Return facts that meet the reliability threshold.

        Args:
            categories:     Optional list of FactCategory values to filter by.
                            None = return all categories.
            min_confidence: Override the store's default threshold.
        """
        threshold = min_confidence if min_confidence is not None else self.confidence_threshold

        facts = [
            f for f in self._facts.values()
            if f.confidence >= threshold
        ]

        if categories:
            facts = [f for f in facts if f.category in categories]
            # Filter to only requested categories.

        # Sort by confidence descending — highest confidence first.
        facts.sort(key=lambda f: f.confidence, reverse=True)

        return facts

    def format_for_injection(
        self,
        categories: Optional[List[str]] = None,
    ) -> str:
        """
        Format reliable facts as a context block for injection.
        Groups by category for readability.
        Returns empty string if no reliable facts exist yet.
        """
        reliable = self.get_reliable_facts(categories=categories)

        if not reliable:
            return ""

        # Group by category.
        by_cat: Dict[str, List[SemanticFact]] = {}
        for fact in reliable:
            cat = fact.category
            if cat not in by_cat:
                by_cat[cat] = []
            by_cat[cat].append(fact)

        lines = [f"SEMANTIC PROFILE — {self.user_id} (distilled from {len(self._facts)} observed patterns):"]
        for cat, facts in by_cat.items():
            lines.append(f"\n[{cat.upper()}]")
            for fact in facts:
                lines.append(fact.format_for_injection())

        return "\n".join(lines)

    def token_count(self) -> int:
        """Count tokens the semantic profile will consume when injected."""
        return len(TOKENISER.encode(self.format_for_injection()))

    def weaken_contradicted_fact(
        self,
        fact_id: str,
        reason: str = ""
    ) -> None:
        """Weaken a specific fact when a new episode contradicts it."""
        if fact_id in self._facts:
            self._facts[fact_id].weaken(reason=reason)
            print(f"  [WEAKEN] Fact {fact_id}: new confidence = "
                  f"{self._facts[fact_id].confidence:.2f}")

    def persist(self, filepath: str) -> None:
        """Save all facts to a JSON file."""
        record = {
            "schema_version":      "1.0",
            "technique":           "semantic_memory",
            "user_id":             self.user_id,
            "created_at":          self.created_at,
            "saved_at":            datetime.now(timezone.utc).isoformat(),
            "distillation_count":  self._distillation_count,
            "confidence_threshold": self.confidence_threshold,
            "stats": {
                "total_facts":      len(self._facts),
                "reliable_facts":   len(self.get_reliable_facts()),
                "profile_tokens":   self.token_count(),
            },
            "facts": {
                fid: asdict(fact)
                for fid, fact in self._facts.items()
            },
        }
        with open(filepath, "w", encoding="utf-8") as f:
            json.dump(record, f, indent=2, ensure_ascii=False)
        print(f"[SemanticStore] Persisted — "
              f"{len(self._facts)} total facts, "
              f"{len(self.get_reliable_facts())} reliable")

    @classmethod
    def load(cls, filepath: str) -> "SemanticMemoryStore":
        """Restore semantic store from a JSON file."""
        with open(filepath, "r", encoding="utf-8") as f:
            record = json.load(f)
        store = cls(
            user_id=record["user_id"],
            confidence_threshold=record["confidence_threshold"],
        )
        for fid, fact_dict in record["facts"].items():
            store._facts[fid] = SemanticFact(**fact_dict)
        store._distillation_count = record["distillation_count"]
        store.created_at = record["created_at"]
        print(f"[SemanticStore] Loaded — "
              f"{len(store._facts)} facts, "
              f"{len(store.get_reliable_facts())} reliable")
        return store

    def print_all_facts(self) -> None:
        """Print all facts with full metadata."""
        print(f"\n{'='*62}")
        print(f"  Semantic Memory Store — User: {self.user_id}")
        print(f"  Total facts: {len(self._facts)} | "
              f"Reliable (>={self.confidence_threshold:.0%}): {len(self.get_reliable_facts())}")
        print(f"  Profile tokens: ~{self.token_count()}")
        print(f"{'='*62}")
        for fid, fact in sorted(self._facts.items(),
                                 key=lambda x: x[1].confidence, reverse=True):
            status = "[INJECT]" if fact.is_reliable(self.confidence_threshold) else "[STORED]"
            print(f"  {status} [{fact.category}] conf={fact.confidence:.2f} "
                  f"obs={fact.observation_count}")
            print(f"    {fact.statement}")
        print(f"{'='*62}\n")


print("SemanticMemoryStore class defined.")

SemanticMemoryStore class defined.


---
## Part 4 — The SemanticMemory Class

Combines the semantic store with a recent buffer and triggers distillation when enough episodes have accumulated.

In [7]:
class SemanticMemory:
    """
    Full semantic memory system for FinCoach.

    Manages:
    - SemanticMemoryStore: the distilled facts about the user
    - Recent buffer: current session conversational flow
    - Distillation trigger: fires when N new episodes have accumulated

    API context structure:
      [system: persona]
      [system: semantic profile]   <- always injected, compact, high signal
      [recent buffer]
    """

    def __init__(
        self,
        session_id: str,
        user_id: str,
        system_prompt: str,
        max_recent_turns: int = 5,
        distill_every_n_episodes: int = 2,
        # Trigger distillation every N episodes.
        # 2 episodes = distil after every 2 sessions.
        # In production: 3-5 episodes is typical — balances freshness vs cost.
        confidence_threshold: float = 0.65,
    ):
        self.session_id                 = session_id
        self.user_id                    = user_id
        self.system_prompt              = system_prompt
        self.max_recent_turns           = max_recent_turns
        self.distill_every_n_episodes   = distill_every_n_episodes

        self.semantic_store = SemanticMemoryStore(
            user_id=user_id,
            confidence_threshold=confidence_threshold,
        )

        self._recent_buffer: List[Dict] = []
        self._pending_episodes: List[Dict] = []
        # Episodes accumulated since the last distillation run.
        # When len >= distill_every_n_episodes, distillation fires.

        self._total_turns = 0
        self.created_at   = datetime.now(timezone.utc).isoformat()

        print(f"[SemanticMemory] Initialised — session: {self.session_id}")
        print(f"  Distil every: {self.distill_every_n_episodes} episodes")

    def add_message(self, role: str, content: str) -> None:
        """Add a message to the recent buffer."""
        if role not in ("user", "assistant"):
            raise ValueError(f"Invalid role '{role}'.")
        msg = {"role": role, "content": content}
        self._recent_buffer.append(msg)
        if len(self._recent_buffer) > self.max_recent_turns * 2:
            self._recent_buffer.pop(0)
        if role == "assistant":
            self._total_turns += 1

    def add_episode(
        self,
        episode_summary: str,
        episode_id: str,
        episode_date: str,
    ) -> None:
        """
        Add a completed episode to the pending queue.
        Triggers distillation if the queue has reached the threshold.

        Call this after close_session() from the EpisodicMemory class.
        In production: call asynchronously after each session ends.
        """
        self._pending_episodes.append({
            "summary":    episode_summary,
            "episode_id": episode_id,
            "date":       episode_date,
        })

        print(f"  [QUEUE] Episode added — "
              f"{len(self._pending_episodes)}/{self.distill_every_n_episodes} "
              f"until distillation")

        if len(self._pending_episodes) >= self.distill_every_n_episodes:
            # Threshold reached — run distillation on all pending episodes.
            self._run_distillation()

    def _run_distillation(self) -> None:
        """
        Distil pending episodes into semantic facts.
        Called automatically when the episode queue reaches the threshold.
        """
        print(f"\n  [DISTIL] Running distillation on "
              f"{len(self._pending_episodes)} episodes...")

        new_facts = distil_episodes_to_facts(
            episode_summaries=self._pending_episodes,
            user_id=self.user_id,
        )

        self.semantic_store.merge_facts(new_facts)
        # Merge: new facts added, existing facts strengthened.

        self._pending_episodes = []
        # Clear the queue — these episodes have been distilled.

    def get_messages_for_api(
        self,
        inject_categories: Optional[List[str]] = None,
    ) -> List[Dict[str, str]]:
        """
        Build the complete API message list.

        Structure:
          [system: persona]
          [system: semantic profile]  <- distilled facts, compact
          [recent buffer]

        Args:
            inject_categories: Optional list of categories to include.
                               None = inject all reliable facts.
        """
        messages = []

        messages.append({"role": "system", "content": self.system_prompt})
        # 1. FinCoach's persona.

        semantic_profile = self.semantic_store.format_for_injection(
            categories=inject_categories
        )
        if semantic_profile:
            messages.append({
                "role": "system",
                "content": (
                    "SEMANTIC PROFILE — distilled from all past sessions:\n"
                    f"{semantic_profile}\n"
                    "Use these patterns to personalise your advice for this user."
                )
            })
            # 2. Semantic profile — always injected when facts are available.

        for msg in self._recent_buffer:
            messages.append(msg)
            # 3. Recent buffer.

        return messages

    def print_stats(self) -> None:
        """Print semantic memory state."""
        reliable = self.semantic_store.get_reliable_facts()
        print(f"\n{'='*58}")
        print(f"  Semantic Memory — Session: {self.session_id}")
        print(f"{'='*58}")
        print(f"  Current session turns  : {self._total_turns}")
        print(f"  Pending episodes       : {len(self._pending_episodes)}")
        print(f"  Distillation threshold : {self.distill_every_n_episodes} episodes")
        print(f"  Total facts stored     : {len(self.semantic_store._facts)}")
        print(f"  Reliable facts         : {len(reliable)}")
        print(f"  Profile tokens         : ~{self.semantic_store.token_count()}")
        print(f"{'='*58}")
        self.semantic_store.print_all_facts()


print("SemanticMemory class defined.")

SemanticMemory class defined.


---
## Part 5 — The FinCoach Agent

In [8]:
FINCOACH_SYSTEM_PROMPT = """You are FinCoach, a personal financial advisor assistant.
You serve users in India who want guidance on savings, investments, budgeting, and financial planning.

Your principles:
- Use the SEMANTIC PROFILE to deeply personalise advice — these are distilled patterns, not guesses.
- If the profile indicates anxiety or hesitation, address it proactively without being asked.
- Apply known preferences and constraints automatically — never recommend what the user has shown they dislike.
- Keep responses concise — 3 to 5 sentences unless detail is requested.
- Never recommend specific stocks.
- Always recommend consulting a SEBI-registered advisor for major decisions."""
# Key instruction: "address anxiety proactively" — this uses the behavioural semantic facts
# to change HOW advice is framed, not just what advice is given.


def chat(
    memory: SemanticMemory,
    user_message: str,
    verbose: bool = True
) -> str:
    """Send one message to FinCoach with semantic memory."""

    memory.add_message(role="user", content=user_message)

    response = client.chat.completions.create(
        model=CHAT_MODEL,
        max_tokens=1024,
        temperature=0.7,
        messages=memory.get_messages_for_api(),
    )

    assistant_reply = response.choices[0].message.content
    memory.add_message(role="assistant", content=assistant_reply)

    if verbose:
        reliable = memory.semantic_store.get_reliable_facts()
        profile_tokens = memory.semantic_store.token_count()
        print(f"[Turn {memory._total_turns}] "
              f"API: {response.usage.prompt_tokens} tokens | "
              f"Semantic profile: {len(reliable)} facts, {profile_tokens} tokens")

    return assistant_reply


print("FinCoach chat() with semantic memory defined.")

FinCoach chat() with semantic memory defined.


---
## Part 6 — Demo: Distillation from Simulated Episodes

We simulate a user who has had several sessions with FinCoach. We feed the episode summaries into the distillation engine and watch semantic facts emerge.

In [9]:
# Simulated episode summaries — representing 4 past sessions with Chiru.
# In production these come from Technique 09's EpisodicStore.
simulated_episodes = [
    {
        "episode_id": "ep_001",
        "date": "2025-03-15",
        "summary": (
            "Chiru discussed his salary of ₹1,20,000 and expenses of ₹60,000. "
            "He expressed anxiety about market conditions and was reluctant to consider "
            "equity mutual funds despite FinCoach's recommendation. He ultimately chose "
            "to keep his FD and asked only about debt options. Session outcome was positive "
            "but Chiru declined all equity suggestions."
        )
    },
    {
        "episode_id": "ep_002",
        "date": "2025-04-10",
        "summary": (
            "Market dip was mentioned early in the session. Chiru immediately became "
            "anxious and asked to move existing savings to an FD. FinCoach explained "
            "that debt mutual funds offer better returns than FDs with low risk. "
            "Chiru needed significant reassurance before accepting the suggestion. "
            "He started a ₹3,000/month SIP in a debt fund only after FinCoach emphasised "
            "that principal was protected. He asked the same safety question three times."
        )
    },
    {
        "episode_id": "ep_003",
        "date": "2025-05-05",
        "summary": (
            "Chiru asked about increasing his SIP amount. When FinCoach suggested "
            "adding an equity component for higher returns over a 20-year horizon, "
            "Chiru declined firmly. He prefers written summaries of advice — he asked "
            "FinCoach to list action items at the end of the session. He consistently "
            "prefers short, direct responses over detailed explanations. Goal remains "
            "retirement at 55 but he is cautious about any instrument with market risk."
        )
    },
    {
        "episode_id": "ep_004",
        "date": "2025-06-01",
        "summary": (
            "Chiru's FD of ₹50,000 matured. He was uncertain about where to deploy the "
            "proceeds. FinCoach recommended a short duration debt fund. Chiru took 3 "
            "sessions worth of questions before committing. He always asks about "
            "worst-case scenarios before making any financial decision. He prefers to "
            "hear the risk of loss quantified. He is building an emergency fund — "
            "currently at 3 months of expenses, targeting 6 months."
        )
    },
]

# Initialise the semantic memory system.
semantic_mem = SemanticMemory(
    session_id="session_sem_demo_001",
    user_id="chiru_001",
    system_prompt=FINCOACH_SYSTEM_PROMPT,
    max_recent_turns=5,
    distill_every_n_episodes=2,  # Distil every 2 episodes.
    confidence_threshold=0.65,
)

print("Adding episodes to trigger distillation...")
print("=" * 65)

# Add episodes — distillation fires automatically every 2 episodes.
for ep in simulated_episodes:
    print(f"\nAdding episode {ep['episode_id']} ({ep['date']})...")
    semantic_mem.add_episode(
        episode_summary=ep["summary"],
        episode_id=ep["episode_id"],
        episode_date=ep["date"],
    )
    time.sleep(0.5)  # Small delay between distillation calls.

print("\n" + "=" * 65)
print("All episodes processed. Semantic profile:")
semantic_mem.print_stats()

[SemanticStore] Initialised for user: chiru_001
[SemanticMemory] Initialised — session: session_sem_demo_001
  Distil every: 2 episodes
Adding episodes to trigger distillation...

Adding episode ep_001 (2025-03-15)...
  [QUEUE] Episode added — 1/2 until distillation

Adding episode ep_002 (2025-04-10)...
  [QUEUE] Episode added — 2/2 until distillation

  [DISTIL] Running distillation on 2 episodes...
  [DISTIL] 2 episodes → 4 semantic facts | tokens: 753
  [MERGE] Distillation #1: +4 new facts, 0 strengthened

Adding episode ep_003 (2025-05-05)...
  [QUEUE] Episode added — 1/2 until distillation

Adding episode ep_004 (2025-06-01)...
  [QUEUE] Episode added — 2/2 until distillation

  [DISTIL] Running distillation on 2 episodes...
  [DISTIL] 2 episodes → 6 semantic facts | tokens: 832
  [MERGE] Distillation #2: +6 new facts, 0 strengthened

All episodes processed. Semantic profile:

  Semantic Memory — Session: session_sem_demo_001
  Current session turns  : 0
  Pending episodes      

---
## Part 7 — Demo: Semantic Memory in Action

Now run a live conversation. The semantic profile is injected on every call. Watch how FinCoach's tone and content differ from a naive agent — it proactively addresses anxiety, avoids equity, and uses the known communication preference.

In [10]:
print("LIVE CONVERSATION with Semantic Memory Active")
print("Profile injected on every turn — FinCoach knows Chiru's patterns")
print("=" * 65)

# Print what the model sees before the first turn.
profile_text = semantic_mem.semantic_store.format_for_injection()
print(f"\nSemantic profile being injected (~{semantic_mem.semantic_store.token_count()} tokens):")
print(profile_text)
print("\n" + "─" * 65)

live_turns = [
    "Hi, I have ₹50,000 to invest. What should I do?",
    # Generic question — but FinCoach should immediately steer toward
    # low-risk options based on the semantic profile, not ask for risk preference.

    "What if markets fall after I invest?",
    # Anxiety trigger — FinCoach should proactively reassure based on
    # the 'market volatility triggers anxiety' semantic fact.

    "Can you give me a clear action plan?",
    # Communication preference — profile says user prefers written,
    # structured action items. FinCoach should provide a clean list.
]

for i, turn in enumerate(live_turns, 1):
    print(f"\n--- Turn {i} ---")
    print(f"User: {turn}")
    response = chat(memory=semantic_mem, user_message=turn, verbose=True)
    print(f"FinCoach: {response}")
    time.sleep(1)

print("\n" + "=" * 65)
print("Observe:")
print("  - Turn 1: FinCoach recommended debt/low-risk without being asked about risk")
print("  - Turn 2: FinCoach proactively addressed the anxiety before being asked to")
print("  - Turn 3: FinCoach gave a structured action list, matching the comm. preference")
print("  All of this came from the semantic profile — no prior context in the session.")

LIVE CONVERSATION with Semantic Memory Active
Profile injected on every turn — FinCoach knows Chiru's patterns

Semantic profile being injected (~228 tokens):
SEMANTIC PROFILE — chiru_001 (distilled from 10 observed patterns):

[BEHAVIOURAL]
- The user exhibits a high level of anxiety regarding market conditions and investment risks. [medium confidence]
- The user always asks about worst-case scenarios before making any financial decision. [medium confidence]

[FINANCIAL]
- The user prefers to invest in fixed deposits and debt options over equity investments. [medium confidence]
- The user is building an emergency fund and has a specific target for its size. [medium confidence]

[RISK]
- The user requires significant reassurance and clarity about the safety of investments before making decisions. [medium confidence]
- The user prefers to avoid market risk in his investments. [medium confidence]
- The user quantifies the risk of loss when considering financial options. [medium confidenc

---
## Part 8 — Fact Evolution: Contradiction and Weakening

In [11]:
print("FACT EVOLUTION DEMO — What happens when patterns change")
print("=" * 65)

# Simulate a new episode where Chiru CONTRADICTS his established pattern.
contradicting_episode = {
    "episode_id": "ep_005",
    "date": "2025-07-15",
    "summary": (
        "Chiru proactively asked about equity mutual funds for the first time. "
        "He mentioned reading about long-term wealth creation and expressed interest "
        "in a small equity SIP allocation. He did not show anxiety — he was curious "
        "and engaged. He agreed to start a ₹1,000/month equity SIP in a large-cap index fund. "
        "This is a significant shift from his previously consistent avoidance of equity."
    )
}

print("Before evolution — high-confidence facts:")
for fact in semantic_mem.semantic_store.get_reliable_facts()[:3]:
    print(f"  conf={fact.confidence:.2f}: {fact.statement[:80]}...")

print(f"\nNew episode added: {contradicting_episode['date']}")
print(f"Content: {contradicting_episode['summary'][:100]}...")

# Add the contradicting episode.
semantic_mem.add_episode(
    episode_summary=contradicting_episode["summary"],
    episode_id=contradicting_episode["episode_id"],
    episode_date=contradicting_episode["date"],
)
# This adds to pending_episodes. With distill_every_n_episodes=2,
# distillation fires at episode 2 of the new batch.
# Add one more to trigger distillation.

confirming_episode = {
    "episode_id": "ep_006",
    "date": "2025-08-10",
    "summary": (
        "Chiru reviewed his equity SIP performance. He is still interested in equity "
        "but cautiously — he limited the allocation to 10% of his investable surplus. "
        "He remains primarily conservative but is showing willingness to explore "
        "long-term equity for retirement. Market volatility came up — he showed "
        "less anxiety than in previous sessions, though still asked about downside protection."
    )
}

semantic_mem.add_episode(
    episode_summary=confirming_episode["summary"],
    episode_id=confirming_episode["episode_id"],
    episode_date=confirming_episode["date"],
)

print("\nAfter evolution — updated facts:")
semantic_mem.semantic_store.print_all_facts()

print("Key observations:")
print("  - Facts about equity avoidance may have lower confidence now")
print("  - New facts about cautious equity interest may have emerged")
print("  - Anxiety facts may have weakened slightly")
print("  - This is semantic memory EVOLVING with the user's behaviour")

FACT EVOLUTION DEMO — What happens when patterns change
Before evolution — high-confidence facts:
  conf=0.80: The user exhibits a high level of anxiety regarding market conditions and invest...
  conf=0.80: The user prefers to invest in fixed deposits and debt options over equity invest...
  conf=0.80: The user requires significant reassurance and clarity about the safety of invest...

New episode added: 2025-07-15
Content: Chiru proactively asked about equity mutual funds for the first time. He mentioned reading about lon...
  [QUEUE] Episode added — 1/2 until distillation
  [QUEUE] Episode added — 2/2 until distillation

  [DISTIL] Running distillation on 2 episodes...
  [DISTIL] 2 episodes → 5 semantic facts | tokens: 793
  [MERGE] Distillation #3: +5 new facts, 0 strengthened

After evolution — updated facts:

  Semantic Memory Store — User: chiru_001
  Total facts: 15 | Reliable (>=65%): 15
  Profile tokens: ~338
  [INJECT] [behavioural] conf=0.80 obs=2
    The user exhibits a hi

---
## Part 9 — Persistence and Cross-Session Recovery

In [12]:
STORE_FILE = "/tmp/fincoach_semantic_store_chiru.json"

semantic_mem.semantic_store.persist(STORE_FILE)
# Saves all facts with full metadata and evolution history.

print("\n--- Restoring in a new session ---\n")

# Create a new SemanticMemory for the next session.
new_session_mem = SemanticMemory(
    session_id="session_sem_002",
    user_id="chiru_001",
    system_prompt=FINCOACH_SYSTEM_PROMPT,
    max_recent_turns=5,
    distill_every_n_episodes=2,
)

# Load the persisted semantic store into the new session.
new_session_mem.semantic_store = SemanticMemoryStore.load(STORE_FILE)

print(f"\nNew session ready. Profile has "
      f"{len(new_session_mem.semantic_store.get_reliable_facts())} reliable facts.")
print(f"Token cost of injected profile: ~{new_session_mem.semantic_store.token_count()} tokens")

# A brand new question — the model has never seen this session's context.
# But the semantic profile gives it full understanding of the user.
new_turn = "I'm thinking about my retirement plan. What should be my next step?"
print(f"\nUser: {new_turn}")
response = chat(memory=new_session_mem, user_message=new_turn, verbose=True)
print(f"FinCoach: {response}")
print("\nObservation: FinCoach gave personalised retirement advice without Chiru")
print("repeating his profile — the semantic store carried all the context.")

[SemanticStore] Persisted — 15 total facts, 15 reliable

--- Restoring in a new session ---

[SemanticStore] Initialised for user: chiru_001
[SemanticMemory] Initialised — session: session_sem_002
  Distil every: 2 episodes
[SemanticStore] Initialised for user: chiru_001
[SemanticStore] Loaded — 15 facts, 15 reliable

New session ready. Profile has 15 reliable facts.
Token cost of injected profile: ~338 tokens

User: I'm thinking about my retirement plan. What should be my next step?
[Turn 1] API: 512 tokens | Semantic profile: 15 facts, 338 tokens
FinCoach: Since you're considering your retirement plan and have shown interest in long-term equity investments, a balanced approach could be beneficial. Start by assessing your current savings and deciding how much you can allocate towards retirement without affecting your emergency fund target. Consider a mix of low-risk options like fixed deposits or debt funds for stability, along with a gradual introduction to equity mutual funds for gr

---
## Key Takeaways

**What you built:** A `SemanticMemory` system with a `SemanticMemoryStore` managing confidence-tracked `SemanticFact` objects, a distillation engine that extracts general truths from episode summaries, fact evolution (strengthening and weakening), category-based injection filtering, and persistence across sessions.

---

### The three things to remember

1. **Semantic memory is distilled from episodic memory — it requires episodes to exist first.** You cannot have semantic memory without episodic memory. The distillation pipeline reads episode summaries and asks: what patterns are true across all these sessions? The quality of semantic facts depends directly on the quality and richness of the episode records. This is why episodic memory (Technique 09) is the natural prerequisite.

2. **Confidence tracking is not cosmetic — it controls what gets injected.** A fact observed once is a hypothesis. Only facts above the confidence threshold are injected into the API call. Low-confidence facts are stored and wait for confirmation. This prevents a single misleading episode from polluting the agent's understanding of the user.

3. **Semantic facts have the highest signal-to-token ratio of any memory layer.** A complete semantic profile for a user — 8-10 general truths — costs 200-300 tokens. That budget delivers more personalisation than 2000 tokens of raw episode summaries. Once the distillation pipeline is built, this is the cheapest way to make the agent feel experienced.

---

### Where the course goes from here

The five core long-term memory types are now complete: vector store, entity, episodic, and semantic. The remaining techniques build on these foundations:

- **Technique 08 (Knowledge Graph)**: relationships between entities — not just facts, but how facts connect
- **Technique 24 (Graphiti)**: temporal knowledge graphs — when facts were true, not just what they are
- **Techniques 12-19**: cognitive architecture patterns — how to orchestrate all memory layers together

---


### FAQ

Q: What is Semantic Memory in agent memory?

A: Semantic Memory extracts generalized, timeless facts from conversations and stores them independently of when or how they were learned. Facts like "user prefers Python over JavaScript" or "the API rate limit is 100 requests per minute" are stored with deduplication and contradiction detection. This mirrors how humans store general knowledge separately from specific experiences. It enables the agent to recall facts without needing to replay entire past conversations.

Q: When should I use Semantic Memory instead of Episodic Memory?

A: Use Semantic Memory when you need to recall reusable facts without the surrounding context. Episodic Memory (technique 09) preserves full interaction records, which is useful for "what happened last time" questions. Semantic memory is better for "what do I know about the user" questions. If your agent is a personal assistant that accumulates user preferences across weeks of interaction, semantic memory keeps facts compact. Episodic memory is better for debugging or auditing past interactions.

Q: What are the limits or failure modes of Semantic Memory?

A: Fact extraction quality depends on the LLM. The model may extract incorrect facts, miss implicit information, or fail to detect contradictions between old and new facts. Deduplication is imperfect: "user likes Python" and "user prefers Python" may be stored as separate facts. Over time, the fact store can accumulate stale information if there is no decay or update mechanism. The extraction step adds latency (200-500ms) and cost per turn.

Q: Can I combine Semantic Memory with another memory technique?

A: Yes. The strongest pairing is with technique 09 (Episodic Memory) to create a dual-memory system. Episodes capture the full experience; semantic memory distills the reusable facts. Add technique 14 (Memory Consolidation) to periodically merge duplicate facts, resolve contradictions, and prune stale entries. This three-layer stack (episodic capture, semantic extraction, periodic consolidation) is the foundation of most production memory systems.

Q: What library or framework can I use to skip the implementation work?

A: Mem0 (technique 25) is built around semantic memory: it automatically extracts and deduplicates facts from conversations. Zep (technique 27) provides managed fact extraction with contradiction detection. Letta/MemGPT (technique 26) stores semantic facts in its core memory block. LangChain does not have a dedicated semantic memory class, but you can build one by combining an LLM extraction chain with a vector store and deduplication logic. Cognee also supports fact extraction pipelines.